# 01 · Rohdaten-Check & erste Sichtung

Zweck dieses Notebooks:
1. **Vor** der Pipeline: ein Roh-CSV stichprobenartig prüfen (Schema, Separator, Auffälligkeiten)
2. **Nach** der Pipeline: die erzeugten Artefakte (`data/processed/`) sichten und die Kernzahlen verifizieren

> Die eigentliche Verarbeitung der 5 GB passiert **nicht** hier, sondern in `scripts/01_preprocess.py`.

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))   # Projektwurzel
import config as cfg

pd.set_option("display.max_columns", 30)
print("Rohdaten-Ordner :", cfg.RAW_DATA_DIR)
print("Output-Ordner   :", cfg.PROCESSED_DIR)

## 1 · Ein Roh-CSV anschauen

Wir lesen nur die ersten Zeilen **einer** Transaktionsdatei — genug, um Schema und Inhalt zu prüfen.

In [ ]:
tx_files = sorted(
    p for p in cfg.RAW_DATA_DIR.glob("*.csv")
    if "trans_num" in p.open(encoding="utf-8", errors="replace").readline()
)
print(f"{len(tx_files)} Transaktionsdateien gefunden")
raw = pd.read_csv(tx_files[0], sep=cfg.SEP, nrows=500, dtype={"cc_num": str, "zip": str})
print(tx_files[0].name, "->", raw.shape)
raw.head(3)

In [ ]:
# Schnellpruefungen
print("Spalten:", list(raw.columns))
print("\nFehlende Werte:", int(raw.isna().sum().sum()))
print("Kategorien:", raw['category'].nunique())
print("Haendler-Praefix 'fraud_' (Faker-Artefakt, NICHT das Label!):",
      raw['merchant'].str.startswith('fraud_').all())

## 2 · Aufbereitete Daten sichten

Erst ausführen, nachdem die Pipeline gelaufen ist (`python scripts/01_preprocess.py`).

In [ ]:
meta = json.loads(cfg.META_PATH.read_text())
print(json.dumps(meta, indent=2))
print(f"\nFraud-Rate: {meta['fraud_rate']:.3%}  <- DIE Kernzahl der Praesentation")

In [ ]:
# Aggregate: Fraud-Rate nach Stunde / Kategorie
by_hour = pd.read_parquet(cfg.AGG_DIR / "by_hour.parquet").sort_values("hour")
by_cat  = pd.read_parquet(cfg.AGG_DIR / "by_category.parquet").sort_values("fraud_rate", ascending=False)

ax = by_hour.plot.bar(x="hour", y="fraud_rate", legend=False, color=cfg.COLOR_FRAUD,
                      figsize=(10, 3), title="Fraud-Rate nach Stunde")
ax.set_xlabel("Stunde"); ax.set_ylabel("Rate")
by_cat.head(8)

In [ ]:
# Maskierte Transaktionen (so sehen die Daten fuer Modellierung & App aus)
parts = sorted(cfg.TRANSACTIONS_DIR.glob("part_*.parquet"))
proc = pd.read_parquet(parts[0])
print(parts[0].name, "->", proc.shape)
proc.head(5)